In [2]:
import sys
sys.path.append("../src")

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
from ipywidgets import interact
import info
import loader
import stats
import plotter

dataset_path = "../dataset/processed/features.csv"
    
try:
    df = loader.load_data(dataset_path)
except FileNotFoundError as e:
    print(f"Error: dataset file ({dataset_path}) not found")
    sys.exit(1)

### Distribuzione incrociata dei tratti linguistici

In [3]:
def select_features(feature1, feature2):
  feature_id1 = df[df["Parameter_Name"] == feature1]["Parameter_ID"].iloc[0]
  feature_id2 = df[df["Parameter_Name"] == feature2]["Parameter_ID"].iloc[0]
  
  qdf = stats.filter(df, "Parameter_ID", [feature_id1, feature_id2]) 
  qdf = qdf[["Language_ID", "Parameter_ID", "Code_Name"]]
  

  # Certe lingue sono registrate per più paesi quindi ci sarebbero dei duplicati
  qdf = qdf.drop_duplicates() 
  qdf = qdf.pivot(
    index='Language_ID', 
    columns='Parameter_ID', 
    values='Code_Name'
  ).dropna()

  qdf = pd.crosstab(qdf[feature_id1], qdf[feature_id2])

  plotter.plot_heatmap(qdf, figsize=(10, 10), annotate=True, title="Inventario Consonanti vs Vocali", xlabel="Inventario vocali", ylabel="Inventario consonanti")

features = stats.get_list_of(df, "Parameter_Name")

interact(
  select_features, 
  feature1=features,
  feature2=features
);


interactive(children=(Dropdown(description='feature1', options=("'Want' Complement Subjects", "'When' Clauses"…